In [0]:
# Databricks notebook: 01_ingest_bronze
# Ingests CSV files from Volume into Bronze tables (schema: 01_bronze)
# Full schemas included – copy and run as is

from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import *

# =====================================================================
# Configuration – use your exact catalog and volume names
# =====================================================================
catalog_name = "retail_catalog"
volume_path = f"/Volumes/{catalog_name}/default/csv_files/raw"

bronze_schema = f"{catalog_name}.01_bronze"
silver_schema = f"{catalog_name}.02_silver"
gold_schema   = f"{catalog_name}.03_gold"

# Create schemas if missing
for schema in [bronze_schema, silver_schema, gold_schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# =====================================================================
# Full explicit schemas (copied from your Fabric code)
# =====================================================================
date_schema = StructType([
    StructField("datekey", IntegerType(), False),
    StructField("fulldate", DateType(), False),
    StructField("year", ShortType(), False),
    StructField("quarternumber", ByteType(), False),
    StructField("quartername", StringType(), False),
    StructField("monthnumber", ByteType(), False),
    StructField("monthname", StringType(), False),
    StructField("weekdaynumber", ByteType(), False),
    StructField("weekdayname", StringType(), False),
    StructField("isweekend", ByteType(), False),
    StructField("yearmonth", StringType(), False),
    StructField("yearmonthnumber", IntegerType(), False),
    StructField("yearquarter", StringType(), False),
    StructField("yearquarternumber", IntegerType(), False),
    StructField("yearweek", StringType(), False),
    StructField("yearweeknumber", IntegerType(), False),
    StructField("isholiday", ByteType(), False)
])

customer_schema = StructType([
    StructField("customerid", IntegerType(), False),
    StructField("fullname", StringType(), False),
    StructField("email", StringType(), False),
    StructField("age", ByteType(), False),
    StructField("gender", StringType(), False),
    StructField("city", StringType(), False),
    StructField("tier", StringType(), False),
    StructField("points", IntegerType(), False),
    StructField("isactive", ByteType(), False),
    StructField("lang", StringType(), False),
    StructField("totalspend", DoubleType(), False),
    StructField("regdate", DateType(), False),
    StructField("annualincome", DoubleType(), False),
    StructField("incomebracket", StringType(), False),
    StructField("education", StringType(), False),
    StructField("maritalstatus", StringType(), False),
    StructField("childrencount", ByteType(), False),
    StructField("loyaltysegment", StringType(), False),
    StructField("satisfactionscore", DoubleType(), False),
    StructField("dayssincelastpurchase", IntegerType(), False),
    StructField("hassubscription", ByteType(), False),
    StructField("preferredcontact", StringType(), False),
    StructField("spendmultiplier", DoubleType(), False)
])

product_schema = StructType([
    StructField("productid", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("brand", StringType(), False),
    StructField("unitcost", DoubleType(), False),
    StructField("unitprice", DoubleType(), False),
    StructField("margin_pct", DoubleType(), False),
    StructField("weight", DoubleType(), False),
    StructField("color", StringType(), False),
    StructField("material", StringType(), False),
    StructField("supplierid", IntegerType(), False),
    StructField("isactive", ByteType(), False),
    StructField("minstock", IntegerType(), False),
    StructField("tax_rate", DoubleType(), False),
    StructField("haswarranty", ByteType(), False),
    StructField("ecofriendly", ByteType(), False),
    StructField("seasonalityfactor", DoubleType(), False),
    StructField("warrantymonths", ByteType(), False),
    StructField("ecoscore", IntegerType(), False),
    StructField("releaseyear", ShortType(), False),
    StructField("skucount", IntegerType(), False),
    StructField("isdiscontinued", ByteType(), False),
    StructField("productrating", DoubleType(), False),
    StructField("stockstatus", StringType(), False)
])

store_schema = StructType([
    StructField("storeid", IntegerType(), False),
    StructField("storename", StringType(), False),
    StructField("city", StringType(), False),
    StructField("type", StringType(), False),
    StructField("staff", ShortType(), False),
    StructField("sizem2", IntegerType(), False),
    StructField("hascafe", ByteType(), False),
    StructField("openingyear", ShortType(), False),
    StructField("region", StringType(), False),
    StructField("renovationyear", ShortType(), False),
    StructField("parkingspots", ShortType(), False),
    StructField("storerating", DoubleType(), False),
    StructField("hasdeliveryservice", ByteType(), False),
    StructField("floornumber", ByteType(), False),
    StructField("distancetocitycenterkm", DoubleType(), False),
    StructField("annualrentcost", DoubleType(), False),
    StructField("storesizemultiplier", DoubleType(), False)
])

promotion_schema = StructType([
    StructField("promoid", IntegerType(), False),
    StructField("promoname", StringType(), False),
    StructField("discount_pct", DoubleType(), False),
    StructField("discount_fixed", DoubleType(), False),
    StructField("type", StringType(), False),
    StructField("isactive", ByteType(), False),
    StructField("minspend", IntegerType(), False),
    StructField("channel", StringType(), False),
    StructField("budget", DoubleType(), False),
    StructField("startdate", DateType(), False),
    StructField("enddate", DateType(), False),
    StructField("targetaudience", StringType(), False),
    StructField("maxdiscountcap", DoubleType(), False),
    StructField("isstackable", ByteType(), False),
    StructField("redemption_rate", DoubleType(), False),
    StructField("coderequired", ByteType(), False),
    StructField("promoupliftfactor", DoubleType(), False)
])

sales_schema = StructType([
    StructField("salesid", LongType(), False),
    StructField("datekey", IntegerType(), False),
    StructField("productid", IntegerType(), False),
    StructField("customerid", IntegerType(), False),
    StructField("storeid", IntegerType(), False),
    StructField("promoid", IntegerType(), False),
    StructField("qty", IntegerType(), False),
    StructField("unitprice", DoubleType(), False),
    StructField("tax_rate", DoubleType(), False),
    StructField("net", DoubleType(), False),
    StructField("payment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("grossvalue", DoubleType(), False),
    StructField("discountamount", DoubleType(), False),
    StructField("taxamount", DoubleType(), False),
    StructField("shipcost", DoubleType(), False),
    StructField("isreturn", ByteType(), False),
    StructField("shipweight", DoubleType(), False),
    StructField("discountapplied", ByteType(), False),
    StructField("returnreason", StringType(), False),
    StructField("deliverydays", ByteType(), False),
    StructField("hour", ByteType(), False)
])

# =====================================================================
# Mapping CSV files to tables
# =====================================================================
csv_mapping = [
    ("dim_date.csv",       "bronze_dimdate",      date_schema),
    ("dim_customer.csv",   "bronze_dimcustomer",  customer_schema),
    ("dim_product.csv",    "bronze_dimproduct",   product_schema),
    ("dim_store.csv",      "bronze_dimstore",     store_schema),
    ("dim_promotion.csv",  "bronze_dimpromotion", promotion_schema),
    ("fact_sales.csv",     "bronze_factsales",    sales_schema),
]

# =====================================================================
# Ingestion loop
# =====================================================================
for filename, table_name, schema in csv_mapping:
    full_path = f"{volume_path}/{filename}"
    try:
        dbutils.fs.ls(full_path)
    except Exception:
        print(f"WARNING: File not found – {full_path}")
        continue

    print(f"Ingesting {filename}...")
    df = spark.read.option("header", "true").schema(schema).csv(full_path)
    df = df.withColumn("_source_file", lit(full_path)) \
           .withColumn("_ingestion_ts", current_timestamp()) \
           .withColumn("_file_name", lit(filename))
    if "fact" in table_name.lower():
        df = df.repartition(4)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.{table_name}")
    print(f"✓ Loaded {df.count():,} rows into {bronze_schema}.{table_name}\n")

print("Bronze layer ingestion completed successfully.")

Ingesting dim_date.csv...
✓ Loaded 1,243 rows into retail_catalog.01_bronze.bronze_dimdate

Ingesting dim_customer.csv...
✓ Loaded 200,001 rows into retail_catalog.01_bronze.bronze_dimcustomer

Ingesting dim_product.csv...
✓ Loaded 2,001 rows into retail_catalog.01_bronze.bronze_dimproduct

Ingesting dim_store.csv...
✓ Loaded 201 rows into retail_catalog.01_bronze.bronze_dimstore

Ingesting dim_promotion.csv...
✓ Loaded 102 rows into retail_catalog.01_bronze.bronze_dimpromotion

Ingesting fact_sales.csv...
✓ Loaded 10,000,000 rows into retail_catalog.01_bronze.bronze_factsales

Bronze layer ingestion completed successfully.
